# Poker Chip Stack Segmentation Training (YOLOv8 + Roboflow)

This notebook is designed to run on **Google Colab** to:
1. Download the dataset from Roboflow (COCO segmentation format)
2. Convert COCO annotations to YOLO segmentation labels
3. Train a YOLOv8 segmentation model
4. Run inference on a few validation images
5. Export the trained model to ONNX for browser use


## 0) Runtime setup

In Colab: go to **Runtime -> Change runtime type -> GPU** for faster training.


In [1]:
# Install required packages:
# - roboflow: dataset download
# - ultralytics: YOLOv8 training/inference/export
# - pycocotools: read COCO JSON annotations
!pip -q install roboflow ultralytics pycocotools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 58.2 MB/s eta 0:00:00


In [2]:
# Optional: verify that GPU is available in Colab.
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 1) Download dataset from Roboflow (COCO Segmentation)

As per Roboflow instructions.


In [4]:
from os import environ
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('yanns-workspace-ufdsd').project('poker-chip-stacks-j395l')
version = project.version(3)
dataset = version.download('coco-segmentation')

print('Dataset downloaded to:', dataset.location)


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Poker-Chip-Stacks-3 in coco-segmentation:: 100%|██████████| 841/841 [00:00<00:00, 4328.58it/s]


Dataset downloaded to: /content/Poker-Chip-Stacks-3


## 2) Convert COCO segmentation annotations -> YOLO segmentation labels

YOLOv8 segmentation expects one `.txt` label per image, where each line is:
`class_id x1 y1 x2 y2 ...` with polygon points normalized to `[0, 1]`.


In [ ]:
import json
import shutil
from pathlib import Path

def convert_coco_split_to_yolo_seg(coco_json_path: Path, images_dir: Path, output_labels_dir: Path):
    """
    Convert a COCO segmentation annotation file into YOLOv8 segmentation label files.

    Args:
        coco_json_path: path to COCO JSON file (instances_*.json)
        images_dir: directory containing split images
        output_labels_dir: destination directory for YOLO .txt labels

    Returns:
        categories_ordered: list of category names in class-id order
    """
    output_labels_dir.mkdir(parents=True, exist_ok=True)

    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    # COCO category ids are not guaranteed to be contiguous, so remap them to 0..N-1.
    categories = sorted(coco['categories'], key=lambda c: c['id'])
    cat_id_to_index = {cat['id']: idx for idx, cat in enumerate(categories)}
    categories_ordered = [cat['name'] for cat in categories]

    images_by_id = {img['id']: img for img in coco['images']}

    # Group annotations per image for efficient writing.
    anns_by_image = {}
    for ann in coco['annotations']:
        anns_by_image.setdefault(ann['image_id'], []).append(ann)

    for image_id, image_info in images_by_id.items():
        img_w = image_info['width']
        img_h = image_info['height']
        img_stem = Path(image_info['file_name']).stem
        label_path = output_labels_dir / f'{img_stem}.txt'

        lines = []
        for ann in anns_by_image.get(image_id, []):
            # Skip crowd instances and malformed/empty segmentations.
            if ann.get('iscrowd', 0) == 1:
                continue

            seg = ann.get('segmentation', [])
            if not isinstance(seg, list) or len(seg) == 0:
                continue

            class_id = cat_id_to_index[ann['category_id']]

            # COCO can store multiple polygons for one object; YOLO expects one polygon per line.
            # We write one line per polygon (common pragmatic conversion strategy).
            for poly in seg:
                if len(poly) < 6 or len(poly) % 2 != 0:
                    continue

                norm_points = []
                for i in range(0, len(poly), 2):
                    x = min(max(poly[i] / img_w, 0.0), 1.0)
                    y = min(max(poly[i + 1] / img_h, 0.0), 1.0)
                    norm_points.extend([x, y])

                line = str(class_id) + ' ' + ' '.join(f'{p:.6f}' for p in norm_points)
                lines.append(line)

        # Write empty file if no valid labels: keeps image/label pairing consistent.
        label_path.write_text('\n'.join(lines))

    return categories_ordered


def build_yolo_dataset_from_roboflow_coco(download_root: Path, yolo_root: Path):
    """
    Build a YOLOv8 segmentation-ready dataset structure from Roboflow COCO export.

    Expected source structure from Roboflow COCO export:
      <download_root>/train/_annotations.coco.json
      <download_root>/valid/_annotations.coco.json
      <download_root>/test/_annotations.coco.json   (optional)

    Output structure:
      <yolo_root>/images/{train,val,test}
      <yolo_root>/labels/{train,val,test}
      <yolo_root>/data.yaml
    """
    split_map = {'train': 'train', 'valid': 'val', 'test': 'test'}
    yolo_root.mkdir(parents=True, exist_ok=True)

    class_names = None

    for coco_split, yolo_split in split_map.items():
        src_images_dir = download_root / coco_split
        src_coco_json = src_images_dir / '_annotations.coco.json'

        if not src_images_dir.exists() or not src_coco_json.exists():
            # test split may be absent depending on export settings
            continue

        dst_images_dir = yolo_root / 'images' / yolo_split
        dst_labels_dir = yolo_root / 'labels' / yolo_split
        dst_images_dir.mkdir(parents=True, exist_ok=True)

        # Copy images while skipping annotation JSON files.
        for p in src_images_dir.iterdir():
            if p.is_file() and p.name != '_annotations.coco.json':
                shutil.copy2(p, dst_images_dir / p.name)

        names = convert_coco_split_to_yolo_seg(
            coco_json_path=src_coco_json,
            images_dir=dst_images_dir,
            output_labels_dir=dst_labels_dir,
        )

        if class_names is None:
            class_names = names

    if class_names is None:
        raise RuntimeError('No valid dataset splits were found in the Roboflow COCO download.')

    # Create YOLO dataset config.
    data_yaml = yolo_root / 'data.yaml'
    data_yaml.write_text(
        '\n'.join([
            f'path: {yolo_root.resolve()}',
            'train: images/train',
            'val: images/val',
            'test: images/test',
            f'nc: {len(class_names)}',
            'names:',
            *[f'  {i}: {name}' for i, name in enumerate(class_names)],
        ])
    )

    return data_yaml, class_names


In [ ]:
# Build YOLO-ready dataset from Roboflow COCO export.
download_root = Path(dataset.location)
yolo_dataset_root = download_root.parent / f"{download_root.name}_yolo_seg"

data_yaml_path, class_names = build_yolo_dataset_from_roboflow_coco(download_root, yolo_dataset_root)

print('YOLO dataset path:', yolo_dataset_root)
print('data.yaml path:', data_yaml_path)
print('Classes:', class_names)


## 3) Train a YOLOv8 segmentation model

We start from pretrained `yolov8n-seg.pt` (nano) for speed.
You can switch to `yolov8s-seg.pt` or bigger models for better quality.


In [ ]:
from ultralytics import YOLO

# Choose a segmentation checkpoint (nano is fastest).
model = YOLO('yolov8n-seg.pt')

# Train with reasonable Colab defaults.
# Increase epochs/imgsz for better performance if needed.
train_results = model.train(
    data=str(data_yaml_path),
    task='segment',
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,  # GPU in Colab
    project='runs',
    name='poker_chip_seg',
    exist_ok=True,
)

print('Training complete. Best weights at:', train_results.save_dir)


## 4) Test inference on a few validation images


In [ ]:
import random
from IPython.display import Image, display

# Load best checkpoint produced by training.
best_weights_path = Path(train_results.save_dir) / 'weights' / 'best.pt'
best_model = YOLO(str(best_weights_path))

val_dir = yolo_dataset_root / 'images' / 'val'
val_images = [p for p in val_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}]

num_samples = min(5, len(val_images))
sample_images = random.sample(val_images, num_samples) if num_samples > 0 else []

print(f'Running inference on {len(sample_images)} validation images...')
pred_results = best_model.predict(
    source=[str(p) for p in sample_images],
    task='segment',
    save=True,
    conf=0.25,
    project='runs',
    name='inference_preview',
    exist_ok=True,
)

# Display generated prediction images (with masks/boxes overlaid).
if pred_results:
    pred_dir = Path(pred_results[0].save_dir)
    print('Prediction output dir:', pred_dir)
    for img_path in sorted(pred_dir.glob('*'))[:num_samples]:
        display(Image(filename=str(img_path)))
else:
    print('No prediction results were generated. Check if validation images exist.')


## 5) Export trained model to ONNX

This generates an ONNX file you can use in browser runtimes (for example with ONNX Runtime Web).


In [ ]:
# Export the trained segmentation model to ONNX.
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=True,
    simplify=True,
)

print('ONNX model exported to:', onnx_path)


In [ ]:
# Optional: copy final artifacts to /content for easy Colab download from file browser.
from pathlib import Path

artifacts = [
    best_weights_path,
    Path(onnx_path),
]

for a in artifacts:
    if a.exists():
        print('Artifact:', a)

print('Done.')
